In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 6.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 100.0 MB/s eta 0:00:0000:01


In [3]:
import os
from getpass import getpass

ROBOFLOW_API_KEY = getpass("Enter Roboflow API key: ")

Enter Roboflow API key:  ········


In [4]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

plantdoc_project = (
    rf.workspace("tomatoes")
      .project("plantdoc-tomatoes")
)

plantdoc = plantdoc_project.version(28).download(
    "yolov8",
    location="/kaggle/working/plantdoc_tomatoes"
)

print("PlantDoc location:")
print(plantdoc.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/plantdoc_tomatoes in yolov8:: 100%|██████████| 2484/2484 [00:00<00:00, 7537.80it/s]

PlantDoc location:
/kaggle/working/plantdoc_tomatoes


In [5]:
tomato_project = (
    rf.workspace("smartgreenfarm")
      .project("tomato-leaf-detection-z3776")
)

versions = tomato_project.versions()

print("Available versions:")

for v in versions:
    print(
        "Version:",
        v.version,
        "| Images:",
        getattr(v, "images", "unknown")
    )

loading Roboflow workspace...
loading Roboflow project...
Available versions:
Version: 3 | Images: 1920
Version: 2 | Images: 1920
Version: 1 | Images: 1910


In [6]:
latest_version = max(
    versions,
    key=lambda v: int(v.version)
)

print(
    "Downloading version:",
    latest_version.version
)

tomato_leaf = latest_version.download(
    "yolov8",
    location="/kaggle/working/tomato_leaf_detection"
)

print("Tomato leaf location:")
print(tomato_leaf.location)


Extracting Dataset Version Zip to /kaggle/working/tomato_leaf_detection in yolov8:: 100%|██████████| 3845/3845 [00:00<00:00, 8753.88it/s]

Tomato leaf location:
/kaggle/working/tomato_leaf_detection


In [7]:
from pathlib import Path

for dataset_name, path in {
    "Tomato Leaf": "/kaggle/working/tomato_leaf_detection",
    "PlantDoc": "/kaggle/working/plantdoc_tomatoes"
}.items():

    
    print(dataset_name)
    

    path = Path(path)

    for p in path.iterdir():
        print(p.name)


Tomato Leaf
test
train
README.dataset.txt
valid
data.yaml
README.roboflow.txt

PlantDoc
test
train
README.dataset.txt
valid
data.yaml
README.roboflow.txt


In [8]:
import yaml

for yaml_path in [
    "/kaggle/working/tomato_leaf_detection/data.yaml",
    "/kaggle/working/plantdoc_tomatoes/data.yaml"
]:

     
    print(yaml_path)
    

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    print("nc:", data.get("nc"))
    print("names:")

    names = data.get("names")

    if isinstance(names, dict):
        for k, v in names.items():
            print(k, ":", v)
    else:
        for i, v in enumerate(names):
            print(i, ":", v)


/kaggle/working/tomato_leaf_detection/data.yaml
nc: 1
names:
0 : Healthy Tomato Leaf

/kaggle/working/plantdoc_tomatoes/data.yaml
nc: 2
names:
0 : Healthy
1 : Unhealthy


In [9]:
from pathlib import Path
import shutil
import yaml

SRC1 = Path("/kaggle/working/tomato_leaf_detection")
SRC2 = Path("/kaggle/working/plantdoc_tomatoes")
OUT = Path("/kaggle/working/tomato_leaf_merged")

if OUT.exists():
    shutil.rmtree(OUT)

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)


def get_source_split(src, split):
    mapping = {
        "train": "train",
        "val": "valid",
        "test": "test"
    }
    return src / mapping[split]


def merge_source(src, prefix):

    for split in ["train", "val", "test"]:

        source_split = get_source_split(src, split)

        image_dir = source_split / "images"
        label_dir = source_split / "labels"

        if not image_dir.exists():
            print(f"Missing: {source_split}")
            continue

        images = [
            p for p in image_dir.iterdir()
            if p.suffix.lower() in
            [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        ]

        print(f"{prefix} | {split}: {len(images)} images")

        for image in images:

            new_image_name = f"{prefix}_{image.name}"
            new_image_path = (
                OUT / "images" / split / new_image_name
            )

            shutil.copy2(image, new_image_path)

            old_label = label_dir / f"{image.stem}.txt"

            new_label = (
                OUT / "labels" / split /
                f"{prefix}_{image.stem}.txt"
            )

            if old_label.exists():

                output_lines = []

                with open(old_label, "r") as f:

                    for line in f:

                        parts = line.strip().split()

                        if len(parts) >= 5:

                            # ALL tomato classes → class 0
                            parts[0] = "0"

                            output_lines.append(
                                " ".join(parts)
                            )

                with open(new_label, "w") as f:
                    f.write("\n".join(output_lines))

            else:
                # Negative image
                new_label.touch()


merge_source(SRC1, "tomato")
merge_source(SRC2, "plantdoc")


# Create YAML
data = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "tomato_leaf"
    }
}

with open(OUT / "data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("\nCreated:")
print(OUT / "data.yaml")

tomato | train: 1680 images
tomato | val: 160 images
tomato | test: 80 images
plantdoc | train: 1100 images
plantdoc | val: 68 images
plantdoc | test: 68 images

Created:
/kaggle/working/tomato_leaf_merged/data.yaml


In [10]:
from collections import Counter

for split in ["train", "val", "test"]:

    image_dir = OUT / "images" / split
    label_dir = OUT / "labels" / split

    images = list(image_dir.glob("*"))
    labels = list(label_dir.glob("*.txt"))

    classes = Counter()
    boxes = 0
    empty = 0

    for label in labels:

        lines = [
            x.strip()
            for x in label.read_text().splitlines()
            if x.strip()
        ]

        if not lines:
            empty += 1
            continue

        for line in lines:
            cls = int(line.split()[0])
            classes[cls] += 1
            boxes += 1

    
    print(split.upper())
    
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Boxes:", boxes)
    print("Classes:", classes)
    print("Empty labels:", empty)


TRAIN
Images: 2780
Labels: 2780
Boxes: 12742
Classes: Counter({0: 12742})
Empty labels: 237

VAL
Images: 228
Labels: 228
Boxes: 966
Classes: Counter({0: 966})
Empty labels: 21

TEST
Images: 148
Labels: 148
Boxes: 368
Classes: Counter({0: 368})
Empty labels: 16


In [11]:
from pathlib import Path
from collections import Counter

OUT = Path("/kaggle/working/tomato_leaf_merged")


print("MERGED TOMATO DATASET CHECK")


# Check dataset exists
print("\nDataset exists:", OUT.exists())

#  Check expected structure
for folder in [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]:
    path = OUT / folder
    print(f"{folder:<20}:", path.exists())

#  Count images by filename prefix
for split in ["train", "val", "test"]:

    image_dir = OUT / "images" / split

    tomato_source = 0
    plantdoc_source = 0

    for image in image_dir.iterdir():

        if not image.is_file():
            continue

        if image.name.startswith("tomato_"):
            tomato_source += 1

        elif image.name.startswith("plantdoc_"):
            plantdoc_source += 1

    
    print(split.upper())
    
    print("Tomato Leaf Detection images :", tomato_source)
    print("PlantDoc images              :", plantdoc_source)
    print("Total                         :", tomato_source + plantdoc_source)


#  Verify all labels are class 0

print("CLASS CHECK")


class_counts = Counter()
empty_labels = 0
total_labels = 0

for split in ["train", "val", "test"]:

    label_dir = OUT / "labels" / split

    for label_file in label_dir.glob("*.txt"):

        total_labels += 1

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            empty_labels += 1
            continue

        for line in lines:
            cls = int(line.split()[0])
            class_counts[cls] += 1

print("Total label files :", total_labels)
print("Empty labels      :", empty_labels)
print("Class distribution:", class_counts)


#  Check YAML

print("YAML")


yaml_file = OUT / "data.yaml"

if yaml_file.exists():
    print(yaml_file.read_text())
else:
    print("data.yaml NOT FOUND")

MERGED TOMATO DATASET CHECK

Dataset exists: True
images/train        : True
images/val          : True
images/test         : True
labels/train        : True
labels/val          : True
labels/test         : True

----------------------------------------
TRAIN
----------------------------------------
Tomato Leaf Detection images : 1680
PlantDoc images              : 1100
Total                         : 2780

----------------------------------------
VAL
----------------------------------------
Tomato Leaf Detection images : 160
PlantDoc images              : 68
Total                         : 228

----------------------------------------
TEST
----------------------------------------
Tomato Leaf Detection images : 80
PlantDoc images              : 68
Total                         : 148

CLASS CHECK
Total label files : 3156
Empty labels      : 274
Class distribution: Counter({0: 14076})

YAML
path: /kaggle/working/tomato_leaf_merged
train: images/train
val: images/val
test: images/test
nam

In [12]:
from pathlib import Path

for split in ["train", "val", "test"]:
    files = list(
        (Path("/kaggle/working/tomato_leaf_merged/images") / split).glob("*")
    )

    tomato = [f for f in files if f.name.startswith("tomato_")]
    plantdoc = [f for f in files if f.name.startswith("plantdoc_")]

    print(
        split,
        "| Tomato:", len(tomato),
        "| PlantDoc:", len(plantdoc),
        "| Total:", len(files)
    )

train | Tomato: 1680 | PlantDoc: 1100 | Total: 2780
val | Tomato: 160 | PlantDoc: 68 | Total: 228
test | Tomato: 80 | PlantDoc: 68 | Total: 148


In [13]:
from roboflow import Roboflow

API_KEY = "TcFKmwaOq8DhYGB7PHnr"

rf = Roboflow(api_key=API_KEY)

project = rf.workspace("deep-learning-assignment-ewyc5").project(

    "weed-detection-d7dau"

)

print(project.versions())

loading Roboflow workspace...
loading Roboflow project...
[<roboflow.core.version.Version object at 0x790af5c99be0>, <roboflow.core.version.Version object at 0x790af5dec800>, <roboflow.core.version.Version object at 0x790af7491f10>, <roboflow.core.version.Version object at 0x790af7490b90>, <roboflow.core.version.Version object at 0x790b0b697170>, <roboflow.core.version.Version object at 0x790b0b696e40>, <roboflow.core.version.Version object at 0x790af5e12420>, <roboflow.core.version.Version object at 0x790af5e12c90>, <roboflow.core.version.Version object at 0x790af5e123c0>, <roboflow.core.version.Version object at 0x790af5e11940>]


In [14]:
versions = project.versions()

for v in versions:
    print(
        "Version:",
        v.version,
        "|",
        "Images:",
        getattr(v, "images", "N/A")
    )

Version: 11 | Images: 4041
Version: 10 | Images: 4041
Version: 9 | Images: 4031
Version: 8 | Images: 4024
Version: 7 | Images: 3921
Version: 6 | Images: 3921
Version: 5 | Images: 3509
Version: 4 | Images: 3509
Version: 3 | Images: 0
Version: 2 | Images: 0


In [15]:
version_number = 11

dataset = project.version(version_number).download("yolov8")

print("Downloaded to:")
print(dataset.location)


Extracting Dataset Version Zip to Weed-Detection-11 in yolov8:: 100%|██████████| 8094/8094 [00:02<00:00, 3664.31it/s]


Downloaded to:
/kaggle/working/Weed-Detection-11


In [16]:
from pathlib import Path
import yaml

ROOT = Path(dataset.location)


print("DATASET STRUCTURE")


for p in ROOT.iterdir():
    print(p)


print("DATA.YAML")


yaml_path = ROOT / "data.yaml"
print(yaml_path.read_text())

with open(yaml_path) as f:
    data = yaml.safe_load(f)


print("CLASSES")


names = data["names"]

if isinstance(names, dict):
    for k, v in names.items():
        print(k, ":", v)
else:
    for i, v in enumerate(names):
        print(i, ":", v)

DATASET STRUCTURE
/kaggle/working/Weed-Detection-11/test
/kaggle/working/Weed-Detection-11/train
/kaggle/working/Weed-Detection-11/README.dataset.txt
/kaggle/working/Weed-Detection-11/valid
/kaggle/working/Weed-Detection-11/data.yaml
/kaggle/working/Weed-Detection-11/README.roboflow.txt

DATA.YAML
names:
- amaranthus palmeri
- amaranthus tuberculatus
- ambrosia artemisiifolia
- eclipta
- eleusine indica
- euphorbia maculata
- ipomoea indica
- mollugo verticillata
- physalis angulata
- portulaca oleracea
- senna obtusifolia
- sida rhombifolia
nc: 12
roboflow:
  license: CC BY 4.0
  project: weed-detection-d7dau
  url: https://universe.roboflow.com/deep-learning-assignment-ewyc5/weed-detection-d7dau/dataset/11
  version: 11
  workspace: deep-learning-assignment-ewyc5
test: ../test/images
train: ../train/images
val: ../valid/images


CLASSES
0 : amaranthus palmeri
1 : amaranthus tuberculatus
2 : ambrosia artemisiifolia
3 : eclipta
4 : eleusine indica
5 : euphorbia maculata
6 : ipomoea ind

In [17]:
from pathlib import Path
from collections import Counter


print("DATASET STATISTICS")


for split in ["train", "valid", "test"]:

    image_dir = ROOT / split / "images"
    label_dir = ROOT / split / "labels"

    if not image_dir.exists():
        print(f"\n{split}: NOT FOUND")
        continue

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]

    labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

    class_counts = Counter()
    boxes = 0
    empty = 0

    for label in labels:
        lines = [
            x.strip()
            for x in label.read_text().splitlines()
            if x.strip()
        ]

        if not lines:
            empty += 1
            continue

        for line in lines:
            parts = line.split()

            if len(parts) >= 5:
                class_counts[int(parts[0])] += 1
                boxes += 1

    print(f"\n{split.upper()}")
    
    print("Images       :", len(images))
    print("Label files  :", len(labels))
    print("Empty labels :", empty)
    print("Boxes        :", boxes)
    print("Classes      :", class_counts)


DATASET STATISTICS

TRAIN
----------------------------------------
Images       : 2433
Label files  : 2433
Empty labels : 1
Boxes        : 3562
Classes      : Counter({6: 689, 1: 532, 9: 515, 5: 365, 2: 307, 7: 296, 11: 243, 3: 178, 0: 163, 10: 115, 4: 103, 8: 56})

VALID
----------------------------------------
Images       : 803
Label files  : 803
Empty labels : 0
Boxes        : 1157
Classes      : Counter({6: 238, 9: 158, 1: 155, 5: 143, 2: 95, 11: 85, 7: 80, 3: 56, 0: 54, 10: 51, 8: 23, 4: 19})

TEST
----------------------------------------
Images       : 805
Label files  : 805
Empty labels : 0
Boxes        : 1125
Classes      : Counter({6: 213, 5: 173, 1: 158, 9: 151, 11: 92, 2: 75, 10: 74, 0: 69, 7: 54, 3: 27, 4: 26, 8: 13})


In [18]:
from pathlib import Path
import shutil
import yaml
from collections import Counter


# PATHS


TOMATO = Path("/kaggle/working/tomato_leaf_merged")
WEED   = Path("/kaggle/working/Weed-Detection-11")
OUT    = Path("/kaggle/working/tomato_weed_13class")

# Remove previous incomplete output
if OUT.exists():
    shutil.rmtree(OUT)

for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)



# CLASSES


WEED_CLASSES = [
    "amaranthus_palmeri",
    "amaranthus_tuberculatus",
    "ambrosia_artemisiifolia",
    "eclipta",
    "eleusine_indica",
    "euphorbia_maculata",
    "ipomoea_indica",
    "mollugo_verticillata",
    "physalis_angulata",
    "portulaca_oleracea",
    "senna_obtusifolia",
    "sida_rhombifolia"
]

NAMES = {
    0: "tomato_leaf"
}

for i, name in enumerate(WEED_CLASSES, start=1):
    NAMES[i] = name


IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}



# COPY TOMATO DATASET
# Structure:
# tomato_leaf_merged/
# ├── images/train
# ├── images/val
# └── images/test


def copy_tomato(split):

    image_dir = TOMATO / "images" / split
    label_dir = TOMATO / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    boxes = 0
    empty = 0

    for image in images:

        new_name = f"tomato_{image.name}"

        shutil.copy2(
            image,
            OUT / "images" / split / new_name
        )

        old_label = label_dir / f"{image.stem}.txt"

        new_label = (
            OUT / "labels" / split /
            f"tomato_{image.stem}.txt"
        )

        if not old_label.exists():
            new_label.touch()
            empty += 1
            continue

        lines = [
            x.strip()
            for x in old_label.read_text().splitlines()
            if x.strip()
        ]

        if not lines:
            new_label.touch()
            empty += 1
            continue

        # Tomato is already class 0
        valid_lines = []

        for line in lines:

            parts = line.split()

            if len(parts) >= 5:
                parts[0] = "0"
                valid_lines.append(" ".join(parts))
                boxes += 1

        new_label.write_text(
            "\n".join(valid_lines)
        )

    return len(images), boxes, empty


#
# COPY WEED DATASET
# Structure:
# Weed-Detection-11/
# ├── train/images
# ├── train/labels
# ├── valid/images
# ├── valid/labels
# └── test/images
#    └── test/labels


def copy_weed(split):

    source_split = {
        "train": "train",
        "val": "valid",
        "test": "test"
    }[split]

    image_dir = WEED / source_split / "images"
    label_dir = WEED / source_split / "labels"

    images = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    boxes = 0
    empty = 0
    classes = Counter()

    for image in images:

        new_name = f"weed_{image.name}"

        shutil.copy2(
            image,
            OUT / "images" / split / new_name
        )

        old_label = label_dir / f"{image.stem}.txt"

        new_label = (
            OUT / "labels" / split /
            f"weed_{image.stem}.txt"
        )

        if not old_label.exists():
            new_label.touch()
            empty += 1
            continue

        lines = [
            x.strip()
            for x in old_label.read_text().splitlines()
            if x.strip()
        ]

        if not lines:
            new_label.touch()
            empty += 1
            continue

        valid_lines = []

        for line in lines:

            parts = line.split()

            if len(parts) < 5:
                continue

            old_class = int(parts[0])

            # Weed class 0-11 → final class 1-12
            new_class = old_class + 1

            parts[0] = str(new_class)

            valid_lines.append(" ".join(parts))

            boxes += 1
            classes[new_class] += 1

        new_label.write_text(
            "\n".join(valid_lines)
        )

    return len(images), boxes, empty, classes



# MERGE



print("MERGING DATASETS")


total = {
    "images": 0,
    "boxes": 0,
    "empty": 0
}

for split in ["train", "val", "test"]:

    print(f"\n{split.upper()}")

    t_images, t_boxes, t_empty = copy_tomato(split)

    w_images, w_boxes, w_empty, w_classes = copy_weed(split)

    total["images"] += t_images + w_images
    total["boxes"] += t_boxes + w_boxes
    total["empty"] += t_empty + w_empty

    print("Tomato images :", t_images)
    print("Tomato boxes  :", t_boxes)

    print("Weed images   :", w_images)
    print("Weed boxes    :", w_boxes)

    print("Total images  :", t_images + w_images)
    print("Total boxes   :", t_boxes + w_boxes)



# CREATE YAML


data = {
    "path": str(OUT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 13,
    "names": NAMES
}

yaml_path = OUT / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(
        data,
        f,
        sort_keys=False
    )



# FINAL VERIFICATION



print("FINAL DATASET VERIFICATION")


all_classes = Counter()

for split in ["train", "val", "test"]:

    image_dir = OUT / "images" / split
    label_dir = OUT / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_dir.glob("*.txt"))

    boxes = 0
    empty = 0
    classes = Counter()

    for label in labels:

        lines = [
            x.strip()
            for x in label.read_text().splitlines()
            if x.strip()
        ]

        if not lines:
            empty += 1
            continue

        for line in lines:

            parts = line.split()

            if len(parts) >= 5:

                cls = int(parts[0])

                if cls not in NAMES:
                    raise ValueError(
                        f"Invalid class {cls} in {label}"
                    )

                classes[cls] += 1
                boxes += 1

    all_classes.update(classes)

    print(f"\n{split.upper()}")
    print("-" * 50)
    print("Images       :", len(images))
    print("Labels       :", len(labels))
    print("Empty labels :", empty)
    print("Boxes        :", boxes)
    print("Classes      :", classes)



print("TOTAL")


print("Images        :", total["images"])
print("Bounding boxes:", total["boxes"])
print("Empty labels  :", total["empty"])

print("\nCLASS DISTRIBUTION")

for cls in sorted(all_classes):
    print(
        f"{cls:2d} : "
        f"{NAMES[cls]:30s} "
        f"{all_classes[cls]}"
    )


print("DATA.YAML")


print(yaml_path.read_text())

print("\nCreated:", OUT)

MERGING DATASETS

TRAIN
Tomato images : 2780
Tomato boxes  : 12742
Weed images   : 2433
Weed boxes    : 3562
Total images  : 5213
Total boxes   : 16304

VAL
Tomato images : 228
Tomato boxes  : 966
Weed images   : 803
Weed boxes    : 1157
Total images  : 1031
Total boxes   : 2123

TEST
Tomato images : 148
Tomato boxes  : 368
Weed images   : 805
Weed boxes    : 1125
Total images  : 953
Total boxes   : 1493

FINAL DATASET VERIFICATION

TRAIN
--------------------------------------------------
Images       : 5213
Labels       : 5213
Empty labels : 238
Boxes        : 16304
Classes      : Counter({0: 12742, 7: 689, 2: 532, 10: 515, 6: 365, 3: 307, 8: 296, 12: 243, 4: 178, 1: 163, 11: 115, 5: 103, 9: 56})

VAL
--------------------------------------------------
Images       : 1031
Labels       : 1031
Empty labels : 21
Boxes        : 2123
Classes      : Counter({0: 966, 7: 238, 10: 158, 2: 155, 6: 143, 3: 95, 12: 85, 8: 80, 4: 56, 1: 54, 11: 51, 9: 23, 5: 19})

TEST
-----------------------------

In [19]:
# YOLO26s — TOMATO LEAF + 12 WEED SPECIES

!pip install -q ultralytics

from ultralytics import YOLO
from pathlib import Path

DATA = "/kaggle/working/tomato_weed_13class/data.yaml"

model = YOLO("yolo26s.pt")

results = model.train(
    data=DATA,

    # Training
    epochs=100,
    imgsz=640,
    batch=-1,

    # Hardware
    device=0,
    workers=4,

    # Optimization
    optimizer="auto",

    # Augmentation
    mosaic=1.0,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,

    # Color augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # Regularization
    weight_decay=0.0005,

    # Validation / checkpointing
    val=True,
    save=True,
    save_period=10,

    # Early stopping
    patience=20,

    # Reproducibility
    seed=42,

    # Output
    project="/kaggle/working/runs",
    name="tomato_weed_yolo26s",
    exist_ok=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/tomato_weed_

In [21]:


BEST = "/kaggle/working/runs/tomato_weed_yolo26s/weights/best.pt"

model = YOLO(BEST)

metrics = model.val(
    data="/kaggle/working/tomato_weed_13class/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0
)


print("TEST PERFORMANCE")


print("mAP50    :", metrics.box.map50)
print("mAP50-95 :", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall   :", metrics.box.mr)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26s summary (fused): 122 layers, 9,470,211 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2249.7±571.5 MB/s, size: 229.3 KB)
val: Scanning /kaggle/working/tomato_weed_13class/labels/test.cache... 953 images, 16 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 953/953 363.4Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 290, len(boxes) = 1493. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 5.6it/s 10.7s0.2s
                   all        953       1493      0.896      0.813      0.871      0.666
           tomato_leaf        132        368      0.794      0.557      0.708      0.561
    amaranth

In [22]:

print("PER-CLASS PERFORMANCE")


names = model.names

for i, name in names.items():

    print(
        f"{i:2d} | "
        f"{name:30s} | "
        f"mAP50-95: {metrics.box.maps[i]:.4f}"
    )

PER-CLASS PERFORMANCE
 0 | tomato_leaf                    | mAP50-95: 0.5614
 1 | amaranthus_palmeri             | mAP50-95: 0.6717
 2 | amaranthus_tuberculatus        | mAP50-95: 0.6720
 3 | ambrosia_artemisiifolia        | mAP50-95: 0.5012
 4 | eclipta                        | mAP50-95: 0.7822
 5 | eleusine_indica                | mAP50-95: 0.8344
 6 | euphorbia_maculata             | mAP50-95: 0.6100
 7 | ipomoea_indica                 | mAP50-95: 0.6385
 8 | mollugo_verticillata           | mAP50-95: 0.4904
 9 | physalis_angulata              | mAP50-95: 0.8406
10 | portulaca_oleracea             | mAP50-95: 0.5742
11 | senna_obtusifolia              | mAP50-95: 0.8620
12 | sida_rhombifolia               | mAP50-95: 0.6239


In [23]:
from pathlib import Path
import shutil
import zipfile


# PATHS


RUN_DIR = Path("/kaggle/working/runs/tomato_weed_yolo26s")
BEST_MODEL = RUN_DIR / "weights" / "best.pt"

OUTPUT_DIR = Path("/kaggle/working/final_model")
OUTPUT_DIR.mkdir(exist_ok=True)


# COPY BEST MODEL


model_dest = OUTPUT_DIR / "tomato_weed_yolo26s_best.pt"

shutil.copy2(BEST_MODEL, model_dest)

print("Best model copied:")
print(model_dest)
print(f"Size: {model_dest.stat().st_size / (1024**2):.2f} MB")


# COPY DATA YAML


yaml_path = Path("/kaggle/working/tomato_weed_13class/data.yaml")

if yaml_path.exists():
    shutil.copy2(
        yaml_path,
        OUTPUT_DIR / "data.yaml"
    )


# COPY TRAINING RESULTS


for filename in [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png"
]:
    src = RUN_DIR / filename

    if src.exists():
        shutil.copy2(src, OUTPUT_DIR / filename)


# ZIP EVERYTHING


zip_path = Path("/kaggle/working/tomato_weed_yolo26s_final.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file in OUTPUT_DIR.rglob("*"):

        if file.is_file():

            zipf.write(
                file,
                arcname=file.relative_to(OUTPUT_DIR)
            )


print("FINAL MODEL PACKAGE")

print(zip_path)
print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")

Best model copied:
/kaggle/working/final_model/tomato_weed_yolo26s_best.pt
Size: 19.39 MB

FINAL MODEL PACKAGE
/kaggle/working/tomato_weed_yolo26s_final.zip
ZIP size: 18.66 MB
